# Paragraph compressor — inference only
Load the trained model, enter text, and inspect its reconstruction. No training or dataset download is needed.

Select this repository's **`.venv` Python kernel**, then run the cells in order. The checkpoint lives under ignored `dump/`, so a Git checkout alone will not include its weights. You can change its path below.

The model can make mistakes. Longer inputs are processed in consecutive 256-token windows. Fewer vectors do not mean fewer storage bytes than the original text.

Set `TOKENS_PER_VECTOR` to **32** to try the strongest passing compression pilot. The default remains **8** for the original model. A 64-token pilot failed the recovery checks and is not offered here. Larger-span models use more parameters; see `docs/latent_memory_compression.md`.

In [ ]:
from pathlib import Path
import os
import sys

# Works when launched from the repository or its notebooks folder.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/models/position_compressor/codec.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the complex_fnn repository.")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import torch
from models.position_compressor.codec import Codec

TOKENS_PER_VECTOR = 8  # Choose 8 (original), 16, or 32.
CHECKPOINTS = {
    8: "dump/position-pattern-5000-clean-v1/last.pt",
    16: "dump/compression-span16-v1/last.pt",
    32: "dump/compression-span32-v1/last.pt",
}
CHECKPOINT = ROOT / CHECKPOINTS[TOKENS_PER_VECTOR]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PRECISION = "bf16" if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else "fp32"
print("Repository:", ROOT)
print("Device:", DEVICE, "| Precision:", PRECISION)

## 1. Load the model once
`Codec` combines the trained encoder, reconstruction decoder and tokenizer. Subsequent cells reuse it. The tokenizer is included in the checkpoint.

In [ ]:
if not CHECKPOINT.is_file():
    raise FileNotFoundError(f"Checkpoint missing: {CHECKPOINT}. Set CHECKPOINT to your saved last.pt.")
codec = Codec.load(CHECKPOINT, device=DEVICE, precision=PRECISION)
print("Loaded:", CHECKPOINT)
print(f"Parameters: {sum(p.numel() for p in codec.model.parameters()):,}")

## 2. Enter your text
Edit `TEXT` and rerun this cell and the cells below. Triple quotes let you paste multiple lines. Leading spaces, tabs and line breaks count when comparing exact recovery.

To use a local UTF-8 file instead, uncomment the file-reading example; `newline=""` preserves its original line endings.

In [ ]:
TEXT = """Amina ordered 17 batteries. Nine arrived; eight are still missing.
Do not send another order yet."""

# with (ROOT / "my_text.txt").open("r", encoding="utf-8", newline="") as file:
#     TEXT = file.read()

# The decoder receives only vectors and length metadata, not TEXT.
memory = codec.encode(TEXT)
reconstruction = codec.decode(memory)
print("INPUT:\n" + TEXT)
print("\nRECONSTRUCTION:\n" + reconstruction)
print("\nEXACT MATCH:", reconstruction == TEXT)

## 3. See exactly what changed
A `repr` display makes invisible characters visible: `\n` means a newline and `\t` means a tab. If reconstruction differs, each changed span is listed below.

In [ ]:
from difflib import SequenceMatcher

print("Input (including whitespace):", repr(TEXT))
print("Output (including whitespace):", repr(reconstruction))
changes = [op for op in SequenceMatcher(None, TEXT, reconstruction, autojunk=False).get_opcodes()
           if op[0] != "equal"]
if not changes:
    print("No differences.")
for kind, a, b, c, d in changes:
    print(f"{kind}: input[{a}:{b}] {TEXT[a:b]!r} -> output[{c}:{d}] {reconstruction[c:d]!r}")

## 4. Inspect the compressed representation
A **token** is a piece of text. A **vector** contains 256 learned numbers (features). Each vector represents up to the selected number of tokens (8, 16, or 32). The payload also stores original token lengths and identities for the matching model and tokenizer.

In [ ]:
sizes = codec.measurements(TEXT, memory)
print(f"{sizes['input_tokens']} input tokens -> {sizes['output_vectors']} vectors")
print(f"{sizes['features_per_vector']} features per vector; {sizes['output_features']} total features")
print("Windows:", sizes["chunks"])
print("Original UTF-8 bytes:", sizes["original_utf8_bytes"])
print("Token-ID bytes (int32):", sizes["token_id_bytes_int32"])
print("Vector bytes:", sizes["vector_bytes"])
print("Length metadata bytes:", sizes["length_metadata_bytes"])
print("Identity metadata bytes:", sizes["identity_metadata_bytes"])
print("Vector shapes:", [tuple(chunk["vectors"].shape) for chunk in memory["chunks"]])

## 5. Save vectors and reconstruct from the saved file
This file contains vectors and metadata, without source text or token IDs. Recovering it requires the matching trained model. Saved files stay under ignored `dump/`. Change `MEMORY_FILE` to keep multiple inputs.

In [ ]:
MEMORY_FILE = ROOT / "dump/inference-playground/memory.pt"
MEMORY_FILE.parent.mkdir(parents=True, exist_ok=True)
torch.save(memory, MEMORY_FILE)
saved_reconstruction = codec.recover_file(MEMORY_FILE)
print("Saved:", MEMORY_FILE)
print("Serialized file bytes:", MEMORY_FILE.stat().st_size)
print("Recovered from file:", repr(saved_reconstruction))
print("Matches this input:", saved_reconstruction == TEXT)
assert saved_reconstruction == reconstruction

## 6. Try several inputs
Edit this list to compare more examples. For detailed differences on a failed example, copy it into `TEXT` above.

In [ ]:
EXAMPLES = [
    """The car is not ready. Do not ship it.....werr45565...548urthnv ,wt3qru09R04IWHTREJ'PRECISION{}..................................
    EXAMPLES = [
The car is not ready. Do not ship it.....werr45565...548urthnv ,wt3qru09R04IWHTREJ'PRECISION{}....................................,
    "Invoice 17019: 17.09, not 19.07.",
    "Hello! Bonjour! مرحبا! 你好! 🙂",
    "First line.\n\tIndented second line.  ",
]
for text in EXAMPLES:
    result = codec.reconstruct(text)
    print("\nINPUT:", repr(text))
    print("OUTPUT:", repr(result["output"]))
    print("Exact:", result["exact_match"], "| Tokens:", result["input_tokens"],
          "| Vectors:", result["output_vectors"])
    ...""",
    "Invoice 17019: 17.09, not 19.07.",
    "Hello! Bonjour! مرحبا! 你好! 🙂",
    "First line.\n\tIndented second line.  ",
]
for text in EXAMPLES:
    result = codec.reconstruct(text)
    print("\nINPUT:", repr(text))
    print("OUTPUT:", repr(result["output"]))
    print("Exact:", result["exact_match"], "| Tokens:", result["input_tokens"],
          "| Vectors:", result["output_vectors"])